In [ ]:
!pip install pandas matplotlib seaborn

In [3]:
# =============================================
# 📦 SQLite Database Monitor for Integrati
# =============================================

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import time
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette("husl")
print("✅ Database monitoring tools loaded!")

# --- Configuration ---
DATA_DIR = Path("data")
MAIN_DB = DATA_DIR / "mcp_plexus_data.sqlite3"
ANALYTICS_DB = DATA_DIR / "analytics.db"

print(f"📁 Data directory: {DATA_DIR.absolute()}")
print(f"🗄️  Main database: {MAIN_DB.absolute()}")
print(f"📊 Analytics database: {ANALYTICS_DB.absolute()}")

print("\n📋 Database Status:")
print(f"   Main DB exists: {MAIN_DB.exists()}")
print(f"   Analytics DB exists: {ANALYTICS_DB.exists()}")

# --- Utility Functions ---
def get_db_connection(db_path):
    try:
        conn = sqlite3.connect(db_path)
        conn.row_factory = sqlite3.Row
        return conn
    except Exception as e:
        print(f"❌ Error connecting to {db_path}: {e}")
        return None

def get_table_names(conn):
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    return [row[0] for row in cursor.fetchall()]

def get_table_info(conn, table_name):
    cursor = conn.cursor()
    cursor.execute(f"PRAGMA table_info({table_name})")
    return cursor.fetchall()

def get_table_data(conn, table_name, limit=10):
    try:
        query = f"SELECT * FROM {table_name}"
        if limit:
            query += f" LIMIT {limit}"
        return pd.read_sql_query(query, conn)
    except Exception as e:
        print(f"❌ Error reading from {table_name}: {e}")
        return pd.DataFrame()

def get_row_count(conn, table_name):
    cursor = conn.cursor()
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    return cursor.fetchone()[0]

print("✅ Database utility functions loaded!")

# --- Explorers ---
def explore_database(db_path, name):
    print(f"🔍 Exploring {name} Database")
    print("=" * 60)
    conn = get_db_connection(db_path)
    if not conn:
        return

    tables = get_table_names(conn)
    print(f"📋 Found {len(tables)} tables:")
    for table in tables:
        print(f"   📊 {table}: {get_row_count(conn, table)} rows")

    print("\n📋 Table Details:")
    for table in tables:
        print(f"\n🔍 Table: {table}\n" + "-" * 40)
        columns = get_table_info(conn, table)
        print("📋 Columns:")
        for col in columns:
            print(f"   • {col[1]} ({col[2]})")
        df = get_table_data(conn, table, limit=5)
        if not df.empty:
            print("\n📊 Sample Data:")
            display(df)
        else:
            print("📊 No data in table")
    conn.close()

# Call these to explore
# explore_database(MAIN_DB, "Main")
# explore_database(ANALYTICS_DB, "Analytics")

# --- Monitoring Class ---
class DatabaseMonitor:
    def __init__(self, db_path):
        self.db_path = db_path
        self.last_counts = {}
        self.initialize_counts()

    def initialize_counts(self):
        conn = get_db_connection(self.db_path)
        if conn:
            for table in get_table_names(conn):
                self.last_counts[table] = get_row_count(conn, table)
            conn.close()

    def check_changes(self):
        conn = get_db_connection(self.db_path)
        if not conn:
            return {}
        changes = {}
        for table in get_table_names(conn):
            current = get_row_count(conn, table)
            previous = self.last_counts.get(table, 0)
            if current != previous:
                changes[table] = {
                    'previous': previous,
                    'current': current,
                    'difference': current - previous
                }
                self.last_counts[table] = current
        conn.close()
        return changes

    def get_latest_data(self, table, limit=5):
        conn = get_db_connection(self.db_path)
        if not conn:
            return pd.DataFrame()
        try:
            cols = get_table_info(conn, table)
            col_names = [c[1] for c in cols]
            if 'id' in col_names:
                query = f"SELECT * FROM {table} ORDER BY id DESC LIMIT {limit}"
            elif 'created_at' in col_names:
                query = f"SELECT * FROM {table} ORDER BY created_at DESC LIMIT {limit}"
            elif 'timestamp' in col_names:
                query = f"SELECT * FROM {table} ORDER BY timestamp DESC LIMIT {limit}"
            else:
                query = f"SELECT * FROM {table} LIMIT {limit}"
            df = pd.read_sql_query(query, conn)
            return df
        except Exception as e:
            print(f"❌ Error getting latest data: {e}")
            return pd.DataFrame()
        finally:
            conn.close()

main_monitor = DatabaseMonitor(MAIN_DB)
analytics_monitor = DatabaseMonitor(ANALYTICS_DB)
print("✅ Monitors initialized!")

# --- Live Monitor ---
def run_live_monitor(duration_minutes=5, check_interval_seconds=10):
    print(f"🔍 Live monitoring for {duration_minutes} min, every {check_interval_seconds} sec")
    start = datetime.now()
    end = start + timedelta(minutes=duration_minutes)

    while datetime.now() < end:
        print(f"\n🕐 {datetime.now().strftime('%H:%M:%S')} - Checking...")

        for name, monitor in [("Main", main_monitor), ("Analytics", analytics_monitor)]:
            changes = monitor.check_changes()
            if changes:
                print(f"\n📊 {name} Database Changes:")
                for table, change in changes.items():
                    print(f"   📈 {table}: {change['previous']} → {change['current']} (+{change['difference']})")
                    df = monitor.get_latest_data(table)
                    if not df.empty:
                        display(df)
            else:
                print(f"✅ No changes in {name} database.")

        time.sleep(check_interval_seconds)

# To start live monitoring:
# run_live_monitor(duration_minutes=5, check_interval_seconds=10)

# --- Quick Status ---
def quick_status_check():
    print("🔍 Quick Status Check")
    for db_path, label in [(MAIN_DB, "Main"), (ANALYTICS_DB, "Analytics")]:
        print(f"\n📊 {label} Database:")
        conn = get_db_connection(db_path)
        if conn:
            for table in get_table_names(conn):
                print(f"   📋 {table}: {get_row_count(conn, table)} rows")
            conn.close()

# quick_status_check()

# --- Visualization ---
def visualize_database_data():
    print("📊 Creating visualizations...")
    for db_path, label, color in [
        (MAIN_DB, "Main", 'skyblue'),
        (ANALYTICS_DB, "Analytics", 'lightcoral')
    ]:
        conn = get_db_connection(db_path)
        if conn:
            tables = get_table_names(conn)
            counts = {t: get_row_count(conn, t) for t in tables}
            conn.close()
            counts = {k: v for k, v in counts.items() if v > 0}
            if counts:
                plt.figure(figsize=(12, 6))
                plt.subplot(1, 2, 1)
                plt.bar(counts.keys(), counts.values(), color=color)
                plt.xticks(rotation=45, ha='right')
                plt.title(f"{label} DB - Rows per Table")
                plt.subplot(1, 2, 2)
                plt.pie(counts.values(), labels=counts.keys(), autopct='%1.1f%%')
                plt.title(f"{label} DB - Data Distribution")
                plt.tight_layout()
                plt.show()

# visualize_database_data()

# --- Custom Query ---
def run_custom_query(db_path, query):
    conn = get_db_connection(db_path)
    if not conn:
        return pd.DataFrame()
    try:
        df = pd.read_sql_query(query, conn)
        return df
    except Exception as e:
        print(f"❌ Query error: {e}")
        return pd.DataFrame()
    finally:
        conn.close()

# 📝 Example:
# df = run_custom_query(MAIN_DB, "SELECT * FROM plexus_user_auth_tokens LIMIT 5")
# display(df)


✅ Database monitoring tools loaded!
📁 Data directory: /home/arunraj/development/integrati/integrati/integrati_backend/data
🗄️  Main database: /home/arunraj/development/integrati/integrati/integrati_backend/data/mcp_plexus_data.sqlite3
📊 Analytics database: /home/arunraj/development/integrati/integrati/integrati_backend/data/analytics.db

📋 Database Status:
   Main DB exists: True
   Analytics DB exists: True
✅ Database utility functions loaded!
✅ Monitors initialized!


In [4]:
df = run_custom_query(MAIN_DB, "SELECT * FROM plexus_user_auth_tokens LIMIT 5")
display(df)


,entity_id,persistent_user_id,token_hash,created_at,last_used_at
0,arun7569raj,uFRwuOFgXpRTLdMTf3GJlzeuJ6g1,4c49fcc6b2948dee4241016cd7eaa92cabdd4dcbb6dee2...,2025-07-30T20:55:50.131418+00:00,2025-07-30T20:55:50.326470+00:00
